In [0]:
# ── PARAMETERS ──────────────────────────────────────────────────────────────
# Update FILE_NAME each time you upload a new file to the Volume.
# Everything else stays constant.

CATALOG   = "clutchlytics"
SCHEMA    = "bronze"
VOLUME    = "nhl_raw"
FILE_NAME = "teams.json"
TABLE     = f"{CATALOG}.{SCHEMA}.raw_nhl_teams"
 
FILE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{FILE_NAME}"
 
print(f"Source : {FILE_PATH}")
print(f"Target : {TABLE}")


In [0]:
# ── READ + PARSE JSON ────────────────────────────────────────────────────────
 
import json
from datetime import datetime, timezone
 
raw_text   = spark.read.text(FILE_PATH)
json_str   = "\n".join([row.value for row in raw_text.collect()])
data       = json.loads(json_str)
 
# ESPN teams structure (with extra "data" wrapper):
# { "data": { "sports": [ { "leagues": [ { "teams": [ { "team": {...} } ] } ] } ] }, "meta": {...} }
teams_raw = data["data"]["sports"][0]["leagues"][0]["teams"]
 
print(f"Teams extracted: {len(teams_raw)}")

In [0]:
# ── FLATTEN TO ROWS ──────────────────────────────────────────────────────────
# Keep all ESPN fields at the top level.
# Add ingestion metadata columns.
 
ingested_at = datetime.now(timezone.utc).isoformat()
 
rows = []
for item in teams_raw:
    team = item.get("team", item)
 
    rows.append({
        # ── ESPN core fields ──
        "team_id":          team.get("id"),
        "uid":              team.get("uid"),
        "slug":             team.get("slug"),
        "abbreviation":     team.get("abbreviation"),
        "display_name":     team.get("displayName"),
        "short_name":       team.get("shortDisplayName"),
        "name":             team.get("name"),
        "nickname":         team.get("nickname"),
        "location":         team.get("location"),
        "color":            team.get("color"),
        "alternate_color":  team.get("alternateColor"),
        "is_active":        team.get("isActive"),
        "is_all_star":      team.get("isAllStar"),
 
        # ── Venue (if present) ──
        "venue_id":         team.get("venue", {}).get("id"),
        "venue_name":       team.get("venue", {}).get("fullName"),
        "venue_city":       team.get("venue", {}).get("address", {}).get("city"),
        "venue_state":      team.get("venue", {}).get("address", {}).get("state"),
 
        # ── Links (logo URL — first logo in array) ──
        "logo_href":        team.get("logos", [{}])[0].get("href") if team.get("logos") else None,
 
        # ── Ingestion metadata ──
        "season":           2026,
        "source_file":      FILE_NAME,
        "ingested_at":      ingested_at,
    })
 
print(f"Rows built: {len(rows)}")
print(f"\nSample row (first team):")
for k, v in rows[0].items():
    print(f"  {k:<20} = {v}")

In [0]:
# ── WRITE TO DELTA ────────────────────────────────────────────────────────────────────────────────────────────────────
# Overwrite on each run — teams reference data, full refresh is fine.

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType

# Define explicit schema to handle None values in venue fields
schema = StructType([
    StructField("team_id", StringType(), True),
    StructField("uid", StringType(), True),
    StructField("slug", StringType(), True),
    StructField("abbreviation", StringType(), True),
    StructField("display_name", StringType(), True),
    StructField("short_name", StringType(), True),
    StructField("name", StringType(), True),
    StructField("nickname", StringType(), True),
    StructField("location", StringType(), True),
    StructField("color", StringType(), True),
    StructField("alternate_color", StringType(), True),
    StructField("is_active", BooleanType(), True),
    StructField("is_all_star", BooleanType(), True),
    StructField("venue_id", StringType(), True),       # nullable
    StructField("venue_name", StringType(), True),     # nullable
    StructField("venue_city", StringType(), True),     # nullable
    StructField("venue_state", StringType(), True),    # nullable
    StructField("logo_href", StringType(), True),
    StructField("season", IntegerType(), False),
    StructField("source_file", StringType(), False),
    StructField("ingested_at", StringType(), False),
])

teams_df = spark.createDataFrame(rows, schema=schema)
 
(
    teams_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE)
)
 
print(f"Written to {TABLE}")

In [0]:
# ── VALIDATE ─────────────────────────────────────────────────────────────────
 
result = spark.sql(f"""
    SELECT
        team_id,
        abbreviation,
        display_name,
        location,
        venue_name,
        venue_city,
        is_active,
        ingested_at
    FROM {TABLE}
    ORDER BY display_name
""")
 
print(f"Total teams in table: {result.count()}")
result.show(32, truncate=False)